# 14b — Time series and honest backtests — Worked answers

Compare reasoning as well as output. Close this file and repeat with changed inputs after 48 hours.

**Prerequisites:** earlier lessons in the course order. **Environment:** Install requirements/core.txt.

[Stage guide](../stages/14.md) · [Course map](../PLAN.md) · [Project](../../projects/stage14/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Build lagged features using past information
- Use chronological and walk-forward splits
- Account for publication times and transaction costs


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Before the exercises: Time ordering and as-of information

Read this in small pieces. Explain each new term in your own words, trace the example on paper, and change one input before continuing. If the arithmetic is unfamiliar, use [the math bridge](../MATH_START_HERE.md).

### Concepts in plain language

A time series contains observations ordered in time. A forecast must use only information available at the decision time. A filing's fiscal year describes its reporting period; its publication timestamp determines when a user could know it.

An as-of join selects the latest record published before a decision, subject to the stated exact-timestamp policy. Walk-forward validation trains on earlier observations and evaluates later ones, repeating across folds. A gap can reduce leakage when targets/features overlap adjacent periods.

Returns are `(new_price-old_price)/old_price`. A strategy chooses a position before the return it earns. Turnover measures position changes and creates trading costs in the teaching model. A shifted signal prevents using the current period's completed return to decide its own position.

### Worked example: follow the values

A 2024 statement published in March 2025 cannot inform a February 2025 decision. A price move from 100 to 110 is +10%, then 110 to 99 is -10%; combined wealth is `1.1*.9=.99`, a 1% loss, not zero.

### Pause and explain

Why can a random split overstate forecasting performance?

<details><summary>Check your reasoning after trying</summary>

It can mix future regimes and overlapping information into training; it fails to reproduce the real chronological decision process.

</details>

### Common mistakes to check

Restatements may be known only later. Preserve historical data vintages. Synthetic random prices establish implementation behavior, not a trading advantage.

**Ready to continue?** Explain the example without reading it, predict what a changed input will do, and identify one invalid or missing input. Then attempt the original notebook exercises. You do not need to finish a whole notebook in one session.


## Lesson

A time series has ordered observations that may depend on earlier values. A return compares prices across time; rolling statistics summarize a trailing window. Autocorrelation measures lagged association. Stationarity refers to distributional properties staying stable over time, not a guarantee that prediction is possible.

Random splitting can leak future regimes or overlapping labels. Use chronological validation or expanding/rolling windows and a gap when label horizons overlap. Every feature needs an availability timestamp. A company’s year-end statement may not be published until months later; joining by fiscal year alone leaks future information. Restatements, survivorship bias, and corporate actions also matter.

A strategy’s signal must precede the return it claims. Include transaction costs, turnover, slippage, and plausible execution assumptions. Compare to a simple baseline and preserve an untouched final period. A profitable synthetic backtest proves implementation mechanics only, not a real trading edge.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
import pandas as pd
prices = pd.Series([100., 105., 102., 110.])
print(prices.pct_change())
print(prices.shift(1).rolling(2).mean())


## Exercise 1: Past-only feature

Return a Series containing the mean of the previous window prices, excluding today; incomplete windows stay NaN.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def lagged_mean(prices, window=2):
    return prices.shift(1).rolling(window).mean()


In [ ]:
values = lagged_mean(pd.Series([10, 20, 100, 200]))
assert values.iloc[2] == 15 and values.iloc[3] == 60
print("Exercise 1: checks passed")


**Why this works:** An unshifted close-price feature may be unavailable when the decision must be made.


## Exercise 2: Chronological split

Return train indices [0,cut-gap) and test [cut,n); require 0<=gap<cut<n.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def temporal_split(n, cut, gap=0):
    if not 0 <= gap < cut < n: raise ValueError("Invalid split")
    return list(range(cut-gap)), list(range(cut,n))


In [ ]:
assert temporal_split(10, 6, 2) == ([0, 1, 2, 3], [6, 7, 8, 9])
print("Exercise 2: checks passed")


**Why this works:** Choose gap from label overlap and information timing, not a fixed magic number.


## Exercise 3: Net strategy return

Given period returns and positions decided before each period, subtract fee*absolute position change from each gross return; initial position is zero.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def net_returns(returns, positions, fee=.001):
    if len(returns) != len(positions): raise ValueError("Length mismatch")
    previous, result = 0, []
    for ret, position in zip(returns, positions):
        result.append(position*ret-fee*abs(position-previous))
        previous = position
    return result


In [ ]:
assert abs(net_returns([.1, .1], [1, 0])[0]-.099) < 1e-12
assert net_returns([.1, .1], [1, 0])[1] == -.001
print("Exercise 3: checks passed")


**Why this works:** This simple proportional-cost model omits spread, slippage, market impact, and financing.


## Independent transfer

Build walk-forward demand/return experiments and an as-of join on filing publication dates. Compare a naive forecast, report per-fold metrics, and document why random splits overstate performance.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Test your independent assignment

The assignment is work you build separately from the small notebook exercises. Use your own Python file under `work/`; notebooks remain useful for exploration and notes.

1. Read the [14b interface and acceptance cases](../assignments/14b.md).
2. Copy the [blank starter](../assignments/starters/14b.py) to `work/assignments/14b.py`, or adapt your existing implementation to the documented interface. Do not overwrite earlier work.
3. Implement one piece, then run this command **in your terminal from the repository root**, with your course environment activated:

```bash
python tools/test_assignment.py 14b --solution work/assignments/14b.py
```

The runner tests the file you specify. It never silently substitutes the reference answer. `NotImplementedError` is expected while the starter is incomplete. Read a failed assertion, calculate its expected answer by hand, fix your implementation, and rerun. See [how to read tests](../assignments/README.md) if pytest is new to you.

After a serious attempt, compare the [worked implementation](../answers/14b.md#14b-t). To check the provided answer explicitly:

```bash
python tools/test_assignment.py 14b --reference
```

**Completion:** pass the relevant automated cases, complete the additional experiment/live-evidence checklist in the assignment guide, and explain your approach without the answer open. An automated pass does not certify a live server, a deployment, a human review, or independent mastery.


## Explain without code

1. Why is fiscal year insufficient?

   The data was not necessarily public at fiscal year end.

2. Does a good backtest establish an edge?

   No. Selection, overfitting, unrealistic execution, and future regime changes can invalidate it.


## Reading and review

- [Primary reference 1](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
